In [0]:
import mlflow
import pyspark.sql.functions as f
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from datetime import datetime, timedelta
from pyspark.sql import Window

In [0]:
dbutils.widgets.text('model_name', 'kp_catalog.hls_ml.hls_ml_demo')
model_name = dbutils.widgets.get('model_name')

dbutils.widgets.text('source_schema', 'kp_catalog.mimic_incr')
source_schema = dbutils.widgets.get('source_schema')

dbutils.widgets.text('target_schema', 'kp_catalog.hls_ml')
target_schema = dbutils.widgets.get('target_schema')

dbutils.widgets.text('retrain_threshold', '.59')
retrain_threshold = float(dbutils.widgets.get('retrain_threshold'))

dbutils.widgets.text('external_location', 's3://one-env-uc-external-location/kp_ml_demo_dev/target/')
external_location = dbutils.widgets.get('external_location')

In [0]:
mlflow.set_registry_uri('databricks-uc')
client = mlflow.tracking.MlflowClient()
fe = FeatureEngineeringClient()

In [0]:
model_details = client.get_model_version_by_alias(model_name, "production")
model_details

### Look Up Production Model

In [0]:
catalog, schema, model = model_name.split('.')

In [0]:
predictions = spark.table(f'{target_schema}.{model}_predictions')

In [0]:
#TODO: stream this
admissions = spark.table(f'{source_schema}.admissions')

max_enc_date = admissions.select(f.max(f.col('admittime'))).collect()[0][0]

w = Window.partitionBy("subject_id").orderBy("admittime")

outcomes = (
  admissions
  # We can't definitively say if anyone from the last 30 days has readmitted in 30 days
  .filter(col('dischtime') < f.lit(max_enc_date - timedelta(days=30)))
  # Calculate the target variable
  .withColumn('last_discharge', f.lag(col('dischtime')).over(w))
  # Calculate if their most recent discharge was within 30 days
 .withColumn('IS_A_READMISSION', f.when(
      f.col('last_discharge') > f.date_trunc('dd', f.col('admittime')) - f.expr('INTERVAL 30 DAYS'), 1
  ).otherwise(0))
  .withColumn('30_DAY_READMISSION', f.coalesce(f.lead('IS_A_READMISSION').over(w), f.lit(0)))
  .select('hadm_id', 'subject_id', 'admittime', 'dischtime', '30_DAY_READMISSION')
  .orderBy(['admittime'], desc=True)
)


In [0]:
compare = (
  predictions
  .join(outcomes.drop('subject_id','admittime','dischtime'), 'hadm_id', 'inner')
  .withColumn('correct', (col('prediction') == col('30_DAY_READMISSION')).cast('int'))
  .select('hadm_id', 'admittime', 'dischtime', '30_DAY_READMISSION', 'model_version')
  .join(spark.table(f'{target_schema}.{model}_predictions').select('hadm_id', 'prediction'), 'hadm_id', 'inner')
  .withColumn('prediction', col('prediction').cast(IntegerType()))
  # .withColumn('admittime', col('admittime') + f.expr(f'INTERVAL {727+84} DAYS'))
  # .withColumn('dischtime', col('dischtime') + f.expr(f'INTERVAL {727+84} DAYS'))
  .write
  .mode('overwrite')
  .option("mergeSchema", "true")
  .saveAsTable(f"{target_schema}.{model}_outcomes")
)


In [0]:
spark.sql(f"ALTER TABLE {target_schema}.{model}_outcomes SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:
df = spark.table(f"{target_schema}.{model}_outcomes")
df.display()

In [0]:
(
  spark.table(f"{target_schema}.{model}_outcomes")
  # .join(spark.table(f'{target_schema}.{model}_predictions').select('Id', 'prediction'), 'Id', 'inner')
  .withColumn('correct', (f.col('prediction') == f.col('30_DAY_READMISSION')).cast('int'))
  .groupBy(f.date_trunc('dd','dischtime').alias('stop_date'), f.col('model_version'))
  .agg(f.mean('correct'),f.count('correct'))
  .withColumn('theshold', f.lit(.55))
).display()

Databricks visualization. Run in Databricks to view.

In [0]:
retrain_model = (
  spark.table(f"{target_schema}.{model}_outcomes")
  # .join(spark.table(f'{target_schema}.{model}_predictions').select('Id', 'prediction'), 'Id', 'inner')
  .withColumn('correct', (f.col('prediction') == f.col('30_DAY_READMISSION')).cast('int'))
  .groupBy(f.date_trunc('dd','dischtime').alias('stop_date'))
  .agg(f.mean('correct').alias('daily_accuracy'))
  .select(f.min('daily_accuracy') < retrain_threshold)
).collect()[0][0]

if retrain_model is None:
  retrain_model = False


dbutils.jobs.taskValues.set('retrain_model', retrain_model)

retrain_model

Another metric that you could measure is SLA - ie. what time were the predictions made available? This would be difficult to measure in the demo, but straight forward in the real world

In [0]:
retrain_model is None